## 一、先认识 Accuracy：总体预测正确率

**Accuracy** 通常翻译为准确率，本课也会用“总体正确率”帮助理解。它表示全部样本中预测正确的比例。

```plain
Accuracy = 预测正确的数量 / 全部样本数量
         = 8 / 10
         = 0.8
```

## 二、混淆矩阵

混淆矩阵是一张对照**真实标签**和**预测结果**的统计表。它不仅统计模型一共预测对了多少，还会把每条预测拆分为四种情况，帮助我们判断模型发生了哪一种错误。

| 字母  | 英文       | 本课中的含义        |
| --- | -------- | ------------- |
| `T` | True     | 模型判断正确        |
| `F` | False    | 模型判断错误        |
| `P` | Positive | 模型预测为正类，即高优先级 |
| `N` | Negative | 模型预测为负类，即普通   |

| 名称  | 实际   | 预测   | 工单业务含义         |
| --- | ---- | ---- | -------------- |
| TP  | 高优先级 | 高优先级 | 紧急工单被正确识别      |
| FN  | 高优先级 | 普通   | 紧急工单被漏掉，风险通常更高 |
| FP  | 普通   | 高优先级 | 普通工单被误报，浪费资源   |
| TN  | 普通   | 普通   | 普通工单被正确识别      |

scikit-learn 输出的二分类混淆矩阵按照下面的方向排列：

```plain
                 预测普通（0）    预测高优先级（1）
实际普通（0）          TN               FP
实际高优先级（1）      FN               TP
```

```plain
实际普通工单数     = TN + FP
实际高优先级工单数 = FN + TP

预测普通工单数     = TN + FN
预测高优先级工单数 = FP + TP

总工单数         = TP + TN + FP + FN
```

1 读取并检查数据

In [19]:
from os import path
from pathlib import Path

import pandas as pd

# 1. 读取课堂数据
# Path 用于表示文件路径，便于在不同操作系统中使用
data_path = Path("data/02_tickets.csv")
# read_csv() 将 CSV 文件读取为 DataFrame
tickets = pd.read_csv(data_path)



2.构造特征并划分数据

In [20]:
from sklearn.model_selection import train_test_split


# 1. 记录长描述特征使用的固定业务阈值
LONG_MESSAGE_THRESHOLD = 120


# 2. 定义可复用的特征构造函数
def add_ticket_features(ticket_frame: pd.DataFrame) -> pd.DataFrame:
    """使用同一套规则构造训练和预测所需的业务特征。"""
    result = ticket_frame.copy()

    # 根据提交小时判断是否属于深夜时段
    result["is_night"] = (
        (result["submit_hour"] >= 22)
        | (result["submit_hour"] < 6)
    ).astype(int)

    # 根据文本长度判断是否属于长描述
    result["is_long_message"] = (
        result["message_length"] >= LONG_MESSAGE_THRESHOLD
    ).astype(int)
    return result


# 3. 为全部原始工单构造业务特征
tickets = add_ticket_features(tickets)

# 4. 指定模型输入列
feature_columns = [
    "channel",
    "customer_level",
    "has_order_id",
    "submit_hour",
    "message_length",
    "is_night",
    "is_long_message",
]

# 5. 拆分特征 X 和标签 y
X = tickets[feature_columns]
y = tickets["is_high_priority"]

# 6. 先划分训练集与临时数据集
X_train, X_temp, y_train, y_temp = train_test_split(
    X,
    y,
    # 40% 进入临时数据集，剩余 60% 作为训练集
    test_size=0.4,
    # 固定随机种子，使每次划分结果一致
    random_state=42,
    # 尽量保持训练集和临时数据集的标签比例一致
    stratify=y,
)

# 7. 再把临时数据平分为验证集和测试集
X_valid, X_test, y_valid, y_test = train_test_split(
    X_temp,
    y_temp,
    # 临时数据的一半作为测试集，即全部数据的 20%
    test_size=0.5,
    # 固定随机种子，使每次划分结果一致
    random_state=42,
    # 尽量保持验证集和测试集的标签比例一致
    stratify=y_temp,
)

3.训练回归模型

In [25]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# 1. 按数据类型划分字段
numeric_features = [
    "has_order_id",
    "submit_hour",
    "message_length",
    "is_night",
    "is_long_message",
]
category_features = ["channel", "customer_level"]

# 2. 数值列：使用中位数填充，再进行标准化
numeric_pipeline = Pipeline([
    (
        # "imputer" 是缺失值填充步骤的名称
        "imputer",
        # 使用每个数值特征的中位数填充该列缺失值
        SimpleImputer(strategy="median"),
    ),
    (
        # "scaler" 是标准化步骤的名称
        "scaler",
        # 将不同数值特征调整到相近的尺度
        StandardScaler(),
    ),
])

# 3. 类别列：使用众数填充，再进行 One-Hot 编码
category_pipeline = Pipeline([
    (
        # "imputer" 是缺失值填充步骤的名称
        "imputer",
        # 使用每个类别特征中出现次数最多的类别填充缺失值
        SimpleImputer(strategy="most_frequent"),
    ),
    (
        # "onehot" 是类别编码步骤的名称
        "onehot",
        # 遇到训练时没有出现过的新类别时不报错
        OneHotEncoder(handle_unknown="ignore"),
    ),
])

# 4. 让不同字段进入各自的预处理流程
preprocessor = ColumnTransformer([
    # numeric_features 中的字段进入数值处理流程
    ("numeric", numeric_pipeline, numeric_features),
    # category_features 中的字段进入类别处理流程
    ("category", category_pipeline, category_features),
])

# 5. 将全部预处理步骤和逻辑回归模型串联起来
ticket_model = Pipeline([
    # 先完成数值特征和类别特征的预处理
    ("preprocessor", preprocessor),
    (
        # 再使用处理后的全部特征训练分类模型
        "classifier",
        # 最多迭代 1000 次，避免模型在尚未收敛时停止
        LogisticRegression(max_iter=1000),
    ),
])

# 6. 使用训练集同时拟合预处理规则和逻辑回归模型
# Pipeline 会先处理 X_train，再使用处理后的特征和 y_train 训练模型
ticket_model.fit(X_train, y_train)

# 7. 在验证集上进行预测
# Pipeline 会自动使用训练阶段学到的预处理规则转换 X_valid
valid_predictions = ticket_model.predict(X_valid)

### 2.1 使用 scikit-learn 输出混淆矩阵

In [35]:
from sklearn.metrics import confusion_matrix

# 使用训练好的模型预测验证集
valid_predictions = ticket_model.predict(X_valid)

print("验证集样本数：", len(y_valid))
# 按标签 0、1 显示真实样本数量
print(y_valid.value_counts().sort_index())

print("混淆矩阵：")
print(
    confusion_matrix(
        # y_true：验证集中的真实标签
        y_valid,
        # y_pred：模型输出的预测标签
        valid_predictions,
        # 固定行列顺序为普通 0、高优先级 1
        labels=[0, 1],  #label:标签
    )
)

验证集样本数： 12
is_high_priority
0    6
1    6
Name: count, dtype: int64
混淆矩阵：
[[5 1]
 [1 5]]


## 三、四个核心分类指标

### 3.1 Accuracy（准确率）：总体预测正确率


当标签比较均衡、不同错误代价接近时，可以主要参考 Accuracy:
```plain
Accuracy = (TP + TN) / (TP + TN + FP + FN)
```

### 3.2 Precision（精确率 / 查准率）：预测为正的结果有多可信

Precision 的分母是**模型预测为高优先级的工单数**。误报处理成本较高时，应优先关注 Precision:
```plain
Precision = TP / (TP + FP)
```

### 3.3 Recall（召回率 / 查全率）：真正的正类找回了多少

Recall 的分母是**实际为高优先级的工单数**。漏报成本较高时，应优先关注 Recall:
```plain
Recall = TP / (TP + FN)
```

### 3.4 F1 Score（F1 分数）：平衡 Precision 与 Recall

F1 是 Precision 和 Recall 的调和平均:
```plain
F1 = 2 × Precision × Recall / (Precision + Recall)
   = 2 × 0.600 × 0.750 / (0.600 + 0.750)
   ≈ 0.667
```

### 3.6 使用 scikit-learn 计算分类指标

In [28]:
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    f1_score,
    precision_score,
    recall_score,
)

In [29]:
# 输出两个类别的完整评估报告
print(
    classification_report(
        # y_true：验证集中的真实标签
        y_valid,
        # y_pred：模型输出的预测标签
        valid_predictions,
        # 固定报告中的类别顺序
        labels=[0, 1],
        # 将标签 0、1 显示为中文类别名称
        target_names=["普通", "高优先级"],
        # 指标结果保留 3 位小数
        digits=3,
        # 指标分母为 0 时返回 0
        zero_division=0,
    )
)

              precision    recall  f1-score   support

          普通      0.833     0.833     0.833         6
        高优先级      0.833     0.833     0.833         6

    accuracy                          0.833        12
   macro avg      0.833     0.833     0.833        12
weighted avg      0.833     0.833     0.833        12



`classification_report()` 中的 `support` 表示各类别的真实样本数；`macro avg` 和 `weighted avg` 本课只需认识。

In [36]:
# 单独计算业务正类“高优先级”的四个核心指标

#accuracy:准确率
valid_accuracy = accuracy_score(y_valid, valid_predictions)

#precision:精确率
valid_precision = precision_score(
    y_valid,
    valid_predictions,
    # 指定标签 1“高优先级”为正类
    pos_label=1,
    zero_division=0,
)
#recall:召回率
valid_recall = recall_score(
    y_valid,
    valid_predictions,
    pos_label=1,
    zero_division=0,
)
#f1: 平衡precision和recal
valid_f1 = f1_score(
    y_valid,
    valid_predictions,
    pos_label=1,
    zero_division=0,
)

In [37]:
print(f"Accuracy：{valid_accuracy:.3f}")
print(f"高优先级 Precision：{valid_precision:.3f}")
print(f"高优先级 Recall：{valid_recall:.3f}")
print(f"高优先级 F1：{valid_f1:.3f}")

Accuracy：0.833
高优先级 Precision：0.833
高优先级 Recall：0.833
高优先级 F1：0.833


## 四、类别不平衡：识别与处理

类别不平衡是指分类数据中不同类别的样本数量或比例存在明显差异。数量较多的类别称为**多数类**，数量较少的类别称为**少数类**。

### 4.1 检查当前训练集的类别分布

In [33]:
label_counts = y_train.value_counts().sort_index()
#normalize=True 返回频率(占比)
label_rates = y_train.value_counts(normalize=True).sort_index()

print("标签数量：")
print(label_counts)
print("标签比例：")
print(label_rates.round(3))

标签数量：
is_high_priority
0    19
1    17
Name: count, dtype: int64
标签比例：
is_high_priority
0    0.528
1    0.472
Name: proportion, dtype: float64


### 4.2 处理原则

`stratify=y` 只让各数据集尽量保持原始标签比例，不会把不平衡数据自动变成 `50% : 50%`。

| 方法                       | 作用                          |
| ------------------------ | --------------------------- |
| 关注 Precision、Recall 和 F1 | 避免只看 Accuracy 而忽略少数类        |
| 使用类别权重或训练集采样             | 让模型训练时更重视少数类                |
| 调整分类阈值                   | 根据业务成本平衡 Precision 与 Recall |

处理时记住：

1. 先确认不平衡是否符合真实业务，并检查少数类数据质量；
2. 权重和采样只用于训练过程，验证集和测试集应保留真实分布；
3. 不要为了“看起来均衡”随意复制数据。

## 五、欠拟合与过拟合

| 现象            | 常见原因           | 改进方向             |
| ------------- | -------------- | ---------------- |
| 训练集和验证集都差     | 欠拟合、特征弱、标签噪声   | 改进特征、检查标签、尝试合适模型 |
| 训练集很好、验证集明显较差 | 过拟合、数据太少、模型太复杂 | 简化模型、增加数据、正则化    |
| 验证集异常地好       | 重复样本或数据泄漏      | 审计划分和预处理流程       |

In [39]:
# 比较训练集和验证集：

train_predictions = ticket_model.predict(X_train)

print("训练集 F1：", f1_score(y_train, train_predictions, zero_division=0))
print("验证集 F1：", f1_score(y_valid, valid_predictions, zero_division=0))

训练集 F1： 0.9696969696969697
验证集 F1： 0.8333333333333334


## 六、数据泄漏

常见泄漏：

- 使用工单关闭后的处理时长预测工单优先级；
- 在全量数据上计算均值后再划分；
- 同一工单的重复记录分别进入训练集和测试集；
- 时间序列任务用未来数据预测过去；
- 人工标签的直接替代字段出现在特征中。

避免泄漏时要记住三个边界：

| 边界   | 正确做法                   |
| ---- | ---------------------- |
| 数据边界 | 先划分训练集、验证集和测试集         |
| 拟合边界 | 填充、编码、标准化和采样规则只在训练集上学习 |
| 使用边界 | 验证集用于选择方案，测试集只用于最终评估   |

## 七、分类阈值与业务取舍


`predict_proba()` 可以输出每条工单属于各类别的概率，概率列的顺序由模型的 `classes_` 决定。例如，`classes_` 为 `[0, 1]` 时，概率 `[0.30, 0.70]` 表示属于普通工单的概率为 `0.30`，属于高优先级工单的概率为 `0.70`。

```plain
高优先级概率 >= 阈值 → 预测为高优先级
高优先级概率 <  阈值 → 预测为普通
```

| 工单  | 真实标签 | 高优先级概率 |
| --- | ---- | ------ |
| A   | 1    | 0.90   |
| B   | 0    | 0.70   |
| C   | 1    | 0.60   |
| D   | 0    | 0.40   |
| E   | 1    | 0.30   |

| 阈值     | 预测为高优先级的工单 | Precision   | Recall      |
| ------ | ---------- | ----------- | ----------- |
| `0.25` | A、B、C、D、E  | `3/5=0.600` | `3/3=1.000` |
| `0.50` | A、B、C      | `2/3≈0.667` | `2/3≈0.667` |
| `0.75` | A          | `1/1=1.000` | `1/3≈0.333` |

降低阈值可以减少漏报，但也可能增加误报；提高阈值可以减少误报，但也可能漏掉更多真正的高优先级工单。

降低阈值后 Recall 提高，提高阈值后 Recall 下降。Precision 不一定持续上升，因为验证集较小，并且剩余预测中仍可能包含误报。

### 7.1 在验证集上比较阈值

In [41]:
# 找到标签 1“高优先级”对应的概率列
positive_class_index = list(ticket_model.classes_).index(1)
valid_probabilities = ticket_model.predict_proba(X_valid)[
    :, positive_class_index
]

In [42]:
# 比较三个候选阈值
for threshold in [0.01, 0.50, 0.95]:
    # 概率达到阈值时预测为 1，否则预测为 0
    threshold_predictions = (valid_probabilities >= threshold).astype(int)

    # 计算标签 1 的 Precision 和 Recall
    threshold_precision = precision_score(
        y_valid, threshold_predictions, pos_label=1, zero_division=0
    )
    threshold_recall = recall_score(
        y_valid, threshold_predictions, pos_label=1, zero_division=0
    )

    print(
            f"阈值={threshold:.2f}，"
            f"Precision={threshold_precision:.3f}，"
            f"Recall={threshold_recall:.3f}"
    )

阈值=0.01，Precision=0.500，Recall=1.000
阈值=0.50，Precision=0.833，Recall=0.833
阈值=0.95，Precision=0.800，Recall=0.667
